# Compile Logical Experiments from Q#

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do you write logical quantum benchmark programs in Q# (`TargetProfile.Adaptive_RI`) and execute them against a `qodec` protocol?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Declare Layer 0 `qodec` instructions as Q# intrinsics (`@Measurement() ... body intrinsic;`).
- Compile SPAM, single-round syndrome, two-block Bell, and multi-round memory programs to QIR via `qsharp.compile`.
- Execute compiled QIR with `run_qir` and `prepare_syndrome_decoder`.

## Before you begin

This course requires `qdk[ec]>=1.33.0`, `qodec>=0.2.0`, `paulimer`, `binar`, and `ipykernel`. Run the cell below to verify your active Python kernel and environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## How Q# Programs Target a `qodec` Layer

Once a `qodec.Qodec` protocol is defined, how do we write and execute high-level logical quantum programs against it?
In QDK-EC (`qdk.qsharp`), a logical Q# program targets **Layer 0** of our `Qodec` protocol via three rules:
1. **Initialize Q# with `TargetProfile.Adaptive_RI`:**
   ```python
   qsharp.init(target_profile=TargetProfile.Adaptive_RI)
   ```
2. **Each Q# `Qubit` is one encoded `C4` block:** Allocating `use block = Qubit();` allocates one 4-physical-qubit $C_4$ block (which holds **2 logical qubits**!).
3. **Declare custom `qodec` instructions as Q# `body intrinsic;` operations:**
   - Unitary or unreturned-flag instructions (`cx_all`, `syndrome`) return `Unit`.
   - Instructions that return classical measurement bits (`measure_z_all`, `measure_x_all`) or a returned flag (`prepare_z_all`, `prepare_x_all`) are decorated with **`@Measurement()`** and return `Result` (for 1 bit) or `(Result, Result)` (for 2 bits).

## Important Subtlety: Returned Flag `Result` vs. Runtime `on_shot_failure`

When `prepare_z_all` has `flags: [reject]` and no `observe` list:
- If you declare `@Measurement() operation prepare_z_all(block : Qubit) : Result { body intrinsic; }` in Q#, the flag bit is **returned directly to your Q# program** as a `Result` (so your Q# program or benchmark harness can inspect `reject == Result.Zero` and count rejected shots explicitly!).
- If instead you declare `operation prepare_z_all(block : Qubit) : Unit { body intrinsic; }`, the Q# program ignores the return value and the simulator's `on_shot_failure="discard"` policy handles flagged shots automatically.

In [2]:
import qodec
import qdk.ec as ec
from _unit import simulation

# Declare Logical C4 Operations in Q# & Compile Benchmark Programs to QIR
from _unit import simulation
simulation.ensure_course_state(globals(), stage="upgraded_all")
from qdk import TargetProfile, qsharp

qsharp.init(target_profile=TargetProfile.Adaptive_RI)

qsharp.eval("""
@Measurement()
operation prepare_z_all(block : Qubit) : Result { body intrinsic; }

@Measurement()
operation prepare_x_all(block : Qubit) : Result { body intrinsic; }

@Measurement()
operation measure_z_all(block : Qubit) : (Result, Result) { body intrinsic; }

@Measurement()
operation measure_x_all(block : Qubit) : (Result, Result) { body intrinsic; }

operation cx_all(control : Qubit, target : Qubit) : Unit { body intrinsic; }

operation syndrome(block : Qubit) : Unit { body intrinsic; }

operation TestSpamZ() : (Result[], Result[]) {
    use block = Qubit();
    let reject = prepare_z_all(block);
    let (left, right) = measure_z_all(block);
    ([reject], [left, right])
}

operation TestSpamX() : (Result[], Result[]) {
    use block = Qubit();
    let reject = prepare_x_all(block);
    let (left, right) = measure_x_all(block);
    ([reject], [left, right])
}

operation TestSyndromeZ() : (Result[], Result[]) {
    use block = Qubit();
    let reject = prepare_z_all(block);
    syndrome(block);
    let (left, right) = measure_z_all(block);
    ([reject], [left, right])
}

operation TestSyndromeX() : (Result[], Result[]) {
    use block = Qubit();
    let reject = prepare_x_all(block);
    syndrome(block);
    let (left, right) = measure_x_all(block);
    ([reject], [left, right])
}

operation TestTwoBlockCX() : (Result[], Result[]) {
    use ctl = Qubit();
    use tgt = Qubit();
    let r_ctl = prepare_x_all(ctl);
    let r_tgt = prepare_z_all(tgt);
    cx_all(ctl, tgt);
    syndrome(ctl);
    syndrome(tgt);
    cx_all(ctl, tgt);
    let (x_c0, x_c1) = measure_x_all(ctl);
    let (z_t0, z_t1) = measure_z_all(tgt);
    ([r_ctl, r_tgt], [x_c0, x_c1, z_t0, z_t1])
}
""")

benchmarks = {
    "SPAM Z": str(qsharp.compile("TestSpamZ()")),
    "SPAM X": str(qsharp.compile("TestSpamX()")),
    "syndrome Z": str(qsharp.compile("TestSyndromeZ()")),
    "syndrome X": str(qsharp.compile("TestSyndromeX()")),
}
two_block_qir = str(qsharp.compile("TestTwoBlockCX()"))

print("Compiled 5 logical Q# benchmark programs to QIR!")
print("Sample QIR declarations from `TestSyndromeZ`:")
for ln in benchmarks["syndrome Z"].splitlines():
    if "declare" in ln or "call" in ln:
        print(" ", ln.strip())

Compiled 5 logical Q# benchmark programs to QIR!
Sample QIR declarations from `TestSyndromeZ`:
  call void @__quantum__rt__initialize(i8* null)
  call void @prepare_z_all(%Qubit* inttoptr (i64 0 to %Qubit*), %Result* inttoptr (i64 0 to %Result*))
  call void @syndrome(%Qubit* inttoptr (i64 0 to %Qubit*))
  call void @measure_z_all(%Qubit* inttoptr (i64 0 to %Qubit*), %Result* inttoptr (i64 1 to %Result*), %Result* inttoptr (i64 2 to %Result*))
  call void @__quantum__rt__tuple_record_output(i64 2, i8* getelementptr inbounds ([4 x i8], [4 x i8]* @0, i64 0, i64 0))
  call void @__quantum__rt__array_record_output(i64 1, i8* getelementptr inbounds ([6 x i8], [6 x i8]* @1, i64 0, i64 0))
  call void @__quantum__rt__result_record_output(%Result* inttoptr (i64 0 to %Result*), i8* getelementptr inbounds ([8 x i8], [8 x i8]* @2, i64 0, i64 0))
  call void @__quantum__rt__array_record_output(i64 2, i8* getelementptr inbounds ([6 x i8], [6 x i8]* @3, i64 0, i64 0))
  call void @__quantum__rt__res

## Executing Compiled Logical Q# Programs with `qdk.simulation.run_qir`

Let's run our compiled Q# benchmarks through `qdk.simulation.run_qir` using `prepare_syndrome_decoder(protocol)`, first noiselessly ($p = 0$) and then at physical noise rate $p = 0.005$.

In [3]:
# Execute Logical Q# Programs with `run_qir` and `prepare_syndrome_decoder`
from _unit import simulation
simulation.ensure_course_state(globals(), stage="upgraded_all")
from qdk import qsharp
from qdk.simulation import NoiseConfig, run_qir
from qdk.simulation.decoders import prepare_syndrome_decoder

# 1. Noiseless check of all 4 single-block benchmarks + 2-block Bell parity test
for label, qir_prog in list(benchmarks.items()) + [("Two-Block Bell CX", two_block_qir)]:
    shots_out = run_qir(
        qir_prog,
        qodec=protocol,
        decoder=prepare_syndrome_decoder,
        shots=20,
        seed=42,
        type="clifford",
        on_shot_failure="discard",
    )
    assert all(
        all(f == qsharp.Result.Zero for f in flags) and all(b == qsharp.Result.Zero for b in readouts)
        for flags, readouts in shots_out
    ), f"Noiseless check failed on {label}"
print("All 5 Q# logical programs produced 100% [Zero, Zero] in noiseless simulation!")

# 2. Noisy run at p = 0.005 on Two-Block Bell CX
noise = NoiseConfig()
for gate in ["h", "x", "y", "z", "cx"]:
    getattr(noise, gate).set_depolarizing(0.005)
noise.mresetz.x = 0.005

noisy_shots = run_qir(
    two_block_qir,
    qodec=protocol,
    decoder=prepare_syndrome_decoder,
    noise=noise,
    shots=200,
    seed=7,
    type="clifford",
    on_shot_failure="discard",
)
accepted = [readouts for flags, readouts in noisy_shots if all(f == qsharp.Result.Zero for f in flags)]
logical_errors = sum(any(b != qsharp.Result.Zero for b in r) for r in accepted)
print(f"\nTwo-Block Bell CX at p=0.005 (200 shots):")
print(f"  Accepted shots (unflagged prep) : {len(accepted)} / {len(noisy_shots)}")
print(f"  Logical Bell parity errors      : {logical_errors} / {len(accepted)} ({logical_errors / len(accepted):.2%})")

All 5 Q# logical programs produced 100% [Zero, Zero] in noiseless simulation!



Two-Block Bell CX at p=0.005 (200 shots):
  Accepted shots (unflagged prep) : 193 / 200
  Logical Bell parity errors      : 44 / 193 (22.80%)


## Multi-Round Logical Memory Experiment (R = 1, 2, 4, 8 Rounds) & Automatic Parity-Check Chaining

In real QEC experiments, we rarely execute just a single round of `syndrome(block)`. Instead, we run **$R$ consecutive rounds** of syndrome extraction (`for _ in 1..rounds { syndrome(block); }`) to measure logical memory lifetime across time ($d_t$).

> **🔬 QEC Researcher Insight — How `qodec` Chains Parity Checks Across Loops Automatically:**
> When running a multi-round memory loop, parity checks must compare stabilizer measurements from one round to the next across the entire sequence of gadgets.
> In QDK-EC, each `syndrome` gadget simply declares its boundary check equations:
> - `in[0].stabilizers[0] + out[0].stabilizers[0] + circuit.readouts[0] = 0`
> - `in[0].stabilizers[1] + out[0].stabilizers[1] + circuit.readouts[1] = 0`
> When your Q# program executes `for _ in 1..rounds { syndrome(block); }`, `qdk.simulation` and `deq` **automatically stitch `out[0]` of round $t$ to `in[0]` of round $t+1$**, creating the complete space-time parity-check graph for any number of rounds $R$!

Run the cell below to compile `RunMemoryRoundsZ(rounds)` for $R \in \{1, 2, 4, 8\}$ rounds and compare multi-round survival at $p = 0.005$ with and without syndrome post-selection.

In [4]:
# Multi-Round Logical Memory Loop (R = 1, 2, 4, 8 Syndrome Rounds)
from _unit import simulation
simulation.ensure_course_state(globals(), stage="upgraded_all")
from qdk import TargetProfile, qsharp

qsharp.init(target_profile=TargetProfile.Adaptive_RI)
qsharp.eval("""
@Measurement()
operation prepare_z_all(block : Qubit) : Result { body intrinsic; }
@Measurement()
operation measure_z_all(block : Qubit) : (Result, Result) { body intrinsic; }
operation syndrome(block : Qubit) : Unit { body intrinsic; }

operation RunMemoryRoundsZ(rounds : Int) : (Result[], Result[]) {
    use block = Qubit();
    let reject = prepare_z_all(block);
    for _ in 1..rounds {
        syndrome(block);
    }
    let (z0, z1) = measure_z_all(block);
    ([reject], [z0, z1])
}
""")

multi_round_qir = {
    f"Memory Z ({r}r)": str(qsharp.compile(f"RunMemoryRoundsZ({r})"))
    for r in [1, 2, 4, 8]
}

sweep_mr_flags = simulation.sample_sweep(
    protocol, multi_round_qir, [5e-3], shots=500, workers=4, max_readout_score=None
)
sweep_mr_post = simulation.sample_sweep(
    protocol, multi_round_qir, [5e-3], shots=500, workers=4, max_readout_score=0.05
)

print(f"{'Benchmark (p=0.005)':<20} | {'Flags-Only LER':<16} | {'Post-Selected LER (<=0.05)':<28} | {'Post-Selected Acceptance'}")
print("-" * 96)
for r_name in multi_round_qir:
    _, c_flg = sweep_mr_flags[r_name][0]
    _, c_pst = sweep_mr_post[r_name][0]
    ler_flg = c_flg["wrong"] / c_flg["accepted"]
    ler_pst = c_pst["wrong"] / c_pst["accepted"]
    acc_pst = c_pst["accepted"] / c_pst["attempted"]
    print(f"{r_name:<20} | {ler_flg:<16.4f} | {ler_pst:<28.4f} | {acc_pst:.1%}")

Benchmark (p=0.005)  | Flags-Only LER   | Post-Selected LER (<=0.05)   | Post-Selected Acceptance
------------------------------------------------------------------------------------------------
Memory Z (1r)        | 0.0203           | 0.0000                       | 94.2%
Memory Z (2r)        | 0.0588           | 0.0045                       | 88.0%
Memory Z (4r)        | 0.0876           | 0.0000                       | 84.0%
Memory Z (8r)        | 0.1237           | 0.0027                       | 74.0%


## Knowledge check

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">measure_z_all(block)</code> return a 2-tuple <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">(Result, Result)</code> in Q# even though <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">block</code> has type <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">Qubit</code>?</summary>

<div style="padding:0.1em 1em;">

Because `block` represents an encoded $C_4$ block containing $k=2$ logical qubits (`observe: [Z_0, Z_1]`). `qodec` maps the length of the instruction's `observe` list directly to the return tuple arity of the `@Measurement()` Q# intrinsic.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;In <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">TestSpamZ()</code>, why did we return <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">([reject], [left, right])</code> and filter rows where <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">all(f == qsharp.Result.Zero for f in flags)</code> instead of relying solely on <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">on_shot_failure='discard'</code>?</summary>

<div style="padding:0.1em 1em;">

Because `prepare_z_all` was declared as `@Measurement() operation prepare_z_all(block : Qubit) : Result`, which consumes the `reject` flag and returns it into the Q# program as a `Result` value. Returning it explicitly lets our benchmark harness count both `attempted` and `accepted` shots to compute the exact shot-rejection rate alongside the logical error rate.

</div>

</details></div>

## Exercise: Execute a Two-Round Q# Logical Memory Experiment

Compile a 2-round logical $Z$-memory experiment in Q# (`TargetProfile.Adaptive_RI`), execute `10` noiseless shots with `run_qir` and `prepare_syndrome_decoder`, and return the number of shots with zero flag/readout errors.

In [ ]:
from qdk import TargetProfile, qsharp
from qdk.simulation import run_qir
from qdk.simulation.decoders import prepare_syndrome_decoder
from _unit import exercise, simulation


@exercise
def noiseless_two_round_memory_shots() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'upgraded_all')
    qsharp.init(target_profile=TargetProfile.Adaptive_RI)
    qsharp.eval('''
    @Measurement() operation prepare_z_all(b : Qubit) : Result { body intrinsic; }
    @Measurement() operation measure_z_all(b : Qubit) : (Result, Result) { body intrinsic; }
    operation syndrome(b : Qubit) : Unit { body intrinsic; }
    operation Memory2R() : (Result[], Result[]) {
        use b = Qubit();
        let r = prepare_z_all(b);
        syndrome(b);
        syndrome(b);
        let (z0, z1) = measure_z_all(b);
        ([r], [z0, z1])
    }
    ''')
    qir = str(qsharp.compile('Memory2R()'))
    # Run 10 noiseless shots with run_qir and return len(shots_out)
    return ...

**Hint**

Call `shots_out = run_qir(qir, qodec=ns['protocol'], decoder=prepare_syndrome_decoder, shots=10, seed=42, type='clifford')` and return `len(shots_out)`.

In [ ]:
from qdk import TargetProfile, qsharp
from qdk.simulation import run_qir
from qdk.simulation.decoders import prepare_syndrome_decoder
from _unit import exercise, simulation


@exercise
def noiseless_two_round_memory_shots() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'upgraded_all')
    qsharp.init(target_profile=TargetProfile.Adaptive_RI)
    qsharp.eval('''
    @Measurement() operation prepare_z_all(b : Qubit) : Result { body intrinsic; }
    @Measurement() operation measure_z_all(b : Qubit) : (Result, Result) { body intrinsic; }
    operation syndrome(b : Qubit) : Unit { body intrinsic; }
    operation Memory2R() : (Result[], Result[]) {
        use b = Qubit();
        let r = prepare_z_all(b);
        syndrome(b);
        syndrome(b);
        let (z0, z1) = measure_z_all(b);
        ([r], [z0, z1])
    }
    ''')
    qir = str(qsharp.compile('Memory2R()'))
    shots_out = run_qir(qir, qodec=ns['protocol'], decoder=prepare_syndrome_decoder, shots=10, seed=42, type='clifford')
    return sum(all(x == qsharp.Result.Zero for x in flags + logical) for flags, logical in shots_out)

When `syndrome(b)` is called in back-to-back rounds, `qdk.simulation` automatically stitches the output block boundary checks of round 1 to the input block boundary checks of round 2, yielding `10` out of `10` deterministic `[Zero, Zero]` readouts in noiseless simulation.

---

### 🤖 COPILOT INSTRUCTOR BRIEF — Module 7
- **Learning Objectives:**
  1. Initialize `qsharp` with `TargetProfile.Adaptive_RI` and declare the 6 Layer-0 `C4` instructions as Q# intrinsics.
  2. Compile four single-block QEC benchmark programs (`TestSpamZ`, `TestSpamX`, `TestSyndromeZ`, `TestSyndromeX`) and one two-block logical entanglement program (`TestTwoBlockCX`) to QIR via `str(qsharp.compile(...))`.
  3. Execute the compiled QIR programs with `qdk.simulation.run_qir` using `prepare_syndrome_decoder(protocol)`.
- **Common Learner Pitfalls:**
  - Forgetting `@Measurement()` on `measure_z_all` or `prepare_z_all`. Without `@Measurement()`, the Q# compiler treats a `Result`-returning intrinsic as a classical function rather than a quantum measurement instruction in QIR.
  - Using `TargetProfile.Base` instead of `TargetProfile.Adaptive_RI`.

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../06-fault-tolerant-gadgets/fault_tolerant_gadgets.workbook.ipynb" title="Previous unit: Upgrade Gadgets for Fault Tolerance" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Upgrade Gadgets for Fault Tolerance</span></span></a><a href="../08-openqasm-integration/openqasm_integration.workbook.ipynb" title="Next unit: Call Qodec Instructions from OpenQASM 3" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Call Qodec Instructions from OpenQASM 3</span></span><span aria-hidden="true">&#8250;</span></a></div>